# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys
from pprint import pprint

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

/Users/pawel.pozorski/Desktop/MLLM-Shap/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cpu


In [4]:
from mllm_shap.connectors import LiquidAudio, ModelConfig, SpectrogramGuidedAligner
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import ExcludePunctuationTokensFilter
from mllm_shap.shap import Explainer, McShapExplainer
from mllm_shap.shap.embeddings import MeanReducer
from mllm_shap.shap.enums import Mode
from mllm_shap.shap.normalizers import AbsSumNormalizer
from mllm_shap.shap.similarity import TfIdfCosineSimilarity
from mllm_shap.utils.audio import display_audio
from mllm_shap.utils.jupyter import display_shap_colors_df, display_shap_colors_df_audio

# Usage

Lets take sample data directly from Hugging Face.

In [5]:
df = pd.read_parquet("hf://datasets/Pawlo77/mllm-shap/single_sentence.parquet")
sample_entry = df.sample(10).iloc[5].to_dict()
del df

pprint(sample_entry)
display_audio(sample_entry["audio__male"][0])

{'audio__female': array([b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xf3\x84\xc4\x00$R\xca\x04\x00y\x86\xb8\x03@\xf5\x1e\xb3.w\x8c\x0c\x8ar\xe0\xf6=3\x01\xe3\xca\x90\xc2d\xf5\xee\xd3 @\x81\x08\x86\x04\x08N\xec\xf2d\xec\x9a|\xc2\x04""\x08\x13\xdc?b\xca!d\xd3\xbd\xae\x06

Define LiquidAudio model (this call loads it up to the memory!).

Create compact explainer that will make initial call and then explain it using Shapley Values approximated using Monte Carlo. Set minimal number of samples for demo purpose, that is only first-order omission ones (where only one token at the time is hidden) and empty sample (if it is not "globally" empty, that is when expandability is not performed on all tokens).

AbsSumNormalizer divides each value by sum of abs shapley values.

In [6]:
os.getpid()

68562

In [7]:
model = LiquidAudio(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.AUDIO
)  # track and generate only audio history
aligner = SpectrogramGuidedAligner(device=device)
shap = McShapExplainer(
    num_samples=-1,
    mode=Mode.CONTEXTUAL,  # use contextual embeddings, default
    # use mean pooling to reduce token embeddings to single embedding per audio, default
    embedding_reducer=MeanReducer(),
    similarity_measure=TfIdfCosineSimilarity(),  # use TF-IDF weighted cosine similarity to compare embeddings
    normalizer=AbsSumNormalizer(),  # use abs sum normalizer to normalize shap values
)
explainer = Explainer(model=model, shap_explainer=shap)

W0505 12:49:51.223000 68562 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


## PR1 sanity check: alignment without attaching audio


By default, alignment creates segments with `audio=b""` to reduce memory usage.
We then attach audio bytes lazily only if we want playback.

In [8]:
# Align once without audio attachment (explicitly)
segments = aligner(
    transcript=sample_entry["sentences"][0],
    audio_content=sample_entry["audio__male"][0],
    attach_audio=False,
)

print(f"Segments: {len(segments)}")
print("All segments have empty audio bytes:", all(seg.audio == b"" for seg in segments))

Segments: 6
All segments have empty audio bytes: True


In [9]:
# Attach audio bytes on demand (lazy materialization)
aligner.attach_audio_to_segments(segments, audio_content=sample_entry["audio__male"][0])
print(
    "After attach: any segment has non-empty audio:",
    any(len(seg.audio) > 0 for seg in segments),
)

# Play first segment to prove bytes exist now
display_audio(segments[0].audio)

After attach: any segment has non-empty audio: True


Create new chat that treats Assistant messages as system, that is will ignore them for shapley values calculation - they will be feed to each prompt as system messages. This significantly reduces number of requests needed for multi turn expandability, yet might not be possible due to business requirements. 

To further reduce number of calls we exclude punctuation tokens.

In [10]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM,  # explain assistant responses as well
    token_filter=ExcludePunctuationTokensFilter(),  # exclude punctuation tokens from shapley values calculation
)

chat.new_turn(Role.USER)
chat.add_text("Who is speaking in the audio?")
chat.end_turn()
chat.new_turn(Role.USER)
chat.add_audio_with_transcript(
    sample_entry["audio__male"][0],
    transcript=sample_entry["sentences"][0],
    aligner=aligner,
    attach_audio=False,  # lazy audio attachment (new default paradigm)
)
chat.end_turn()

## PR1 sanity check: attach audio to chat segments on demand


If you want per-segment playback later, attach bytes only when needed.

In [11]:
# Attach audio bytes to chat segments lazily (for playback/debug)
# NOTE: Keeping per-segment bytes attached makes chat copies heavier;
# leave this off unless you really need playback at this point.
ATTACH_SEGMENT_AUDIO_FOR_DEBUG = False

if not getattr(chat, "_audio_segments", None):
    raise ValueError(
        "chat._audio_segments is empty; did add_audio_with_transcript() run?"
    )

audio_turn = max(chat._audio_segments.keys())
print("Audio segment turns available:", sorted(chat._audio_segments.keys()))
print("Selected turn:", audio_turn)

if ATTACH_SEGMENT_AUDIO_FOR_DEBUG:
    print("Attaching per-segment audio bytes (debug)...")
    chat.attach_audio_to_segments(
        aligner,
        audio_content=sample_entry["audio__male"][0],
        turn_number=audio_turn,
    )
    print("Attached audio bytes to chat segments for turn:", audio_turn)
else:
    print("Skipping per-segment audio attachment (debug disabled).")

Audio segment turns available: [2]
Selected turn: 2
Skipping per-segment audio attachment (debug disabled).


Let's have a look at chat representation:

In [12]:
pprint(chat.get_conversation())

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  is,  speaking,  in,  the,  audio, ?, <|im_end|>, \n...', shap_values=None)],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, SYSTEM], content='<|im_start|>, user, \n', shap_values=None),
  ChatEntry(content_type=1, roles=[USER, USER, ..., USER, USER], content='Audio bytes of total length 0', shap_values=None),
  ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM], content='<|im_end|>, \n', shap_values=None)]]


In [13]:
chat._audio_segments[2]

[AudioSegment(token='Who', start=0.081, end=0.281, dur=0.200s),
 AudioSegment(token='created', start=0.326, end=0.763, dur=0.436s),
 AudioSegment(token='the', start=0.803, end=0.883, dur=0.080s),
 AudioSegment(token='South', start=0.978, end=1.224, dur=0.246s),
 AudioSegment(token='Beach', start=1.264, end=1.525, dur=0.261s),
 AudioSegment(token='Diet?', start=1.548, end=1.909, dur=0.361s)]

Representation is a list of list of ConversationEntry - so it can be accessed as  chat.get_conversation()[turn_number][message_number].{field}

Let's calculate shapley values for current conversation.

Verbose=False saves memory. Generation kwargs allows to customize model interference - here we limit it to 32 tokens and change text_temperature from default 0.0 to 0.2. 

In [14]:
# --- Rerun-safe cleanup (run this before re-running SHAP) ---
import gc
import torch


def _cuda_mem(msg: str) -> None:
    if not torch.cuda.is_available():
        print(msg, "(cuda not available)")
        return
    torch.cuda.synchronize()
    a = torch.cuda.memory_allocated() / 1024**2
    r = torch.cuda.memory_reserved() / 1024**2
    print(f"{msg}: allocated={a:.1f}MB reserved={r:.1f}MB")


_cuda_mem("Before cleanup")

# 0) Clear IPython output caches that can keep old objects alive across reruns
try:
    from IPython import get_ipython

    ip = get_ipython()
    if ip is not None:
        for k in ("_", "__", "___"):
            ip.user_ns.pop(k, None)
        # Out history (key culprit for "rerun but memory doesn't drop")
        if "_oh" in ip.user_ns and hasattr(ip.user_ns["_oh"], "clear"):
            ip.user_ns["_oh"].clear()
        if "Out" in ip.user_ns and hasattr(ip.user_ns["Out"], "clear"):
            ip.user_ns["Out"].clear()
except Exception:
    pass

# 1) Drop previous SHAP cache(s) if present (this is where big GPU tensors tend to live)
for obj_name in ("explained_chat", "chat"):
    obj = globals().get(obj_name, None)
    if obj is not None and getattr(obj, "cache", None) is not None:
        obj.cache = None

# 2) Ensure we didn't materialize per-segment audio bytes on chats BEFORE SHAP
for obj_name in ("explained_chat", "chat"):
    obj = globals().get(obj_name, None)
    if obj is not None and getattr(obj, "_audio_segments", None):
        for _, segs in obj._audio_segments.items():
            for s in segs:
                try:
                    s.audio = b""
                except Exception:
                    pass

# 3) Drop previous result objects so reruns don't keep multiple graphs alive
for name in (
    "result",
    "explained_chat",
    "explained_chat_conversation",
    "audio_bytes_list",
    "df_audio",
    "normalized_sv",
    "total_n_calls",
):
    if name in globals():
        globals()[name] = None

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

_cuda_mem("After cleanup")

Before cleanup (cuda not available)
After cleanup (cuda not available)


In [15]:
generation_kwargs = {
    "max_new_tokens": 32,
    "model_config": ModelConfig(text_temperature=0.2),
}

result = explainer(
    chat=chat,
    verbose=False,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2026-05-05 12:49:56,002 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2026-05-05 12:49:58,697 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 12 (up to 4095 additional calls)


Monte Carlo SHAP:   0%|          | 0/13 [00:00<?, ?it/s]

2026-05-05 12:50:29,909 - mllm_shap.shap.base._generate_responses - INFO - Generation stats: processed=13 cache_hits=0 cache_misses=13 skipped_filtered=0 model_elapsed_ms=31200.10
2026-05-05 12:50:29,909 - mllm_shap.shap.base.shap_explainer - INFO - Sampling stats: candidates=14 yielded=13 skipped(full_or_empty)=1 skipped(invalid)=0 skipped(duplicates)=0 elapsed_ms=31208.26


Result has now following fields available:

- full_chat - chat with base response (generated based on whole entry) with set cache and calculated shapley values
- source_chat - original chat feed to the explainer
- history - history of all chats used

In [16]:
import gc
import torch

if result is None:
    raise ValueError("`result` is None. Run the SHAP cell above first.")

# Keep what we need for the notebook UI
total_n_calls = getattr(result, "total_n_calls", None)
explained_chat = result.full_chat
explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

# Snapshot normalized SHAP values to CPU for later plotting, then drop the heavy cache to prevent rerun OOM
normalized_sv = None
try:
    normalized_sv = explained_chat.shap.normalized_values
    if torch.is_tensor(normalized_sv):
        normalized_sv = normalized_sv.detach().cpu()
except Exception:
    normalized_sv = None

if getattr(explained_chat, "cache", None) is not None:
    explained_chat.cache = None

# Also drop the result wrapper (full_chat is kept above)
result = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

print("total_n_calls:", total_n_calls)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  is,  speaking,  in,  the,  audio, ?, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, SYSTEM], content='<|im_start|>, user, \n', shap_values=[nan, nan, nan]),
  ChatEntry(content_type=1, roles=[USER, USER, ..., USER, USER], content='Audio bytes of total length 0', shap_values=[0.10528658330440521, 0.097999207675457, ..., nan, nan]),
  ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM], content='<|im_end|>, \n', shap_values=[nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, SYSTEM], content='<|im_start|>, assistant, \n', shap_values=[nan, nan, nan]),
  ChatEntry(content_type=1, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content='Audio bytes of total length 77680', shap_values=[nan, nan, ..., nan, nan]),
  ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM], content='<|im_end|>, \n', shap_valu

In [17]:
explained_chat_conversation[1][1].display()

BY: USER
AUDIO CONTENT:


Encoding audio tokens in seperation creates weird non-continual audio description.

Model was set to return just text tokens, so chat history has only text tokens. We can see that in the json representation inside ConversationEntry shap_values field is now populated. Nan values indicated that this token wasn't taken into calculation scope. As expected, we have 3 not-nan tokens. Let's see them.

In [18]:
user_entry = explained_chat_conversation[0][0]

display_shap_colors_df(
    pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values)),
        columns=["Token", "Shapley Value"],
    )
)

,Token,Shapley Value
0,<|im_start|>,nan
1,user,nan
2,,nan
3,Who,0.088169
4,is,0.081353
5,speaking,0.026177
6,in,0.092381
7,the,0.092564
8,audio,0.080791
9,?,nan


In [19]:
# Show SHAP values per aligned audio segment (robust to turn indexing)
from mllm_shap.connectors.enums import ModalityFlag

# 1) Find the first audio entry that has SHAP values
audio_entry = None
for turn in explained_chat_conversation:
    for entry in turn:
        if (
            entry.content_type == ModalityFlag.AUDIO.value
            and entry.shap_values is not None
        ):
            audio_entry = entry
            break
    if audio_entry is not None:
        break

if audio_entry is None:
    raise ValueError(
        "No audio ChatEntry with SHAP values found in explained_chat_conversation."
    )

# 2) Prefer the aligner segments stored on the chat (lets us lazily materialize bytes)
audio_bytes_list = None
if getattr(explained_chat, "_audio_segments", None):
    audio_turn_key = max(explained_chat._audio_segments.keys())
    segments_for_turn = explained_chat._audio_segments.get(audio_turn_key)
    if segments_for_turn:
        # Ensure we have actual bytes for playback; SHAP values don't depend on it, but the widget does.
        if all(len(seg.audio) == 0 for seg in segments_for_turn):
            explained_chat.attach_audio_to_segments(
                aligner,
                audio_content=sample_entry["audio__male"][0],
                turn_number=audio_turn_key,
            )
        audio_bytes_list = [seg.audio for seg in segments_for_turn]

# 3) Fallback: use the raw bytes stored in the ChatEntry (may be empty if lazy attachment was used)
if audio_bytes_list is None:
    audio_bytes_list = [
        c for c in audio_entry.content if isinstance(c, (bytes, bytearray))
    ]

shap_values = list(audio_entry.shap_values or [])
n = min(len(audio_bytes_list), len(shap_values))
if n == 0:
    raise ValueError("No audio segments or SHAP values to display.")

df_audio = pd.DataFrame(
    list(zip(audio_bytes_list[:n], shap_values[:n])),
    columns=["Audio", "Shapley Value"],
)
display_shap_colors_df_audio(df_audio)